
# 11_classify_final_manifest

Fast Gemma 31B metadata extraction for the final Commission file manifest.

For each successfully processed document:

1. resolve its processed `text_path`;
2. save a persistent classification snippet containing the first 15k and last 10k characters;
3. send only that snippet plus current manifest metadata to Gemma 31B;
4. request a minimal structured JSON containing candidate identifiers/dates only;
5. checkpoint results incrementally while keeping document-level parallel inference; and
6. export the enriched manifest to `output/final_manifest/final_commission_manifest_classified.xlsx`.


The purpose of this notebook is speed and simple manual verification. Gemma is not asked for explanations, evidence excerpts, confidence scores, Commission internal document numbers, or field-by-field assessments. The final manifest remains the master table; Gemma only adds a small set of candidate fields.


In [1]:

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from typing import Any
import asyncio
import hashlib
import json
import os
import re

import pandas as pd
import httpx
from tqdm.auto import tqdm

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    """Find the repository root containing both data/ and output/."""
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "output").is_dir():
            return candidate
    if start == Path("/mnt/data") or Path("/mnt/data").exists():
        return start
    raise FileNotFoundError(
        "Could not find the project root. Run inside the repository containing "
        "data/ and output/, or set PROJECT_ROOT explicitly."
    )


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
OUTPUT_DIR = PROJECT_ROOT / "output" / "final_manifest"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Every Gemma input is persisted here first.
LOG_DIR = PROJECT_ROOT / "logs" / "final_manifest"
LOG_DIR.mkdir(parents=True, exist_ok=True)

SNIPPET_DIR = LOG_DIR / "classification_inputs"
SNIPPET_DIR.mkdir(parents=True, exist_ok=True)


def resolve_manifest_path() -> Path:
    candidates = [
        OUTPUT_DIR / "final_commission_file_manifest.xlsx",
        PROJECT_ROOT / "final_commission_file_manifest.xlsx",
        NOTEBOOK_DIR / "final_commission_file_manifest.xlsx",
        Path("/mnt/data/final_commission_file_manifest.xlsx"),
        Path("/mnt/data/final_commission_file_manifest(3).xlsx"),
    ]
    for path in candidates:
        if path.is_file():
            return path.resolve()
    searched = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(
        "Could not locate final_commission_file_manifest.xlsx. Searched:\n" + searched
    )


MANIFEST_PATH = resolve_manifest_path()

OUTPUT_XLSX = OUTPUT_DIR / "final_commission_manifest_classified.xlsx"
CHECKPOINT_JSONL = LOG_DIR / "final_commission_manifest_classified.jsonl"

# ---------------------------------------------------------------------
# Local Gemma / vLLM
# ---------------------------------------------------------------------
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")
MODEL = os.getenv("VLLM_MODEL", "google/gemma-4-31B-it")

RUN_MODE = os.getenv("RUN_MODE", "update").strip().lower()
VALID_RUN_MODES = {"update", "redo_all", "rebuild_excel"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}")

# Parallel inference: keep this at the level your vLLM setup comfortably supports.
CONCURRENT_REQUESTS = 4
REQUEST_TIMEOUT_SECONDS = 3600
MAX_RETRIES = 2
TEMPERATURE = 0.0

# Output is tiny now: just five candidate arrays.
MAX_OUTPUT_TOKENS = 700

# Set to e.g. 20 for an end-to-end test; None = all pending jobs.
TEST_LIMIT: int | None = None

# ---------------------------------------------------------------------
# Fast fixed-size snippets
# ---------------------------------------------------------------------
HEAD_CHARS = 15_000
TAIL_CHARS = 10_000

# Increment this whenever the prompt/schema/snippet policy changes.
PROMPT_VERSION = "commission_manifest_identity_fast_v3"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Snippet directory:", SNIPPET_DIR)
print("Output workbook:", OUTPUT_XLSX)
print("Checkpoint:", CHECKPOINT_JSONL)
print("Model:", MODEL)
print("RUN_MODE:", RUN_MODE)
print("CONCURRENT_REQUESTS:", CONCURRENT_REQUESTS)
print(f"Snippet policy: first {HEAD_CHARS:,} + last {TAIL_CHARS:,} characters")


PROJECT_ROOT: /home/edik/projects/ec_cjeu
Manifest: /home/edik/projects/ec_cjeu/output/final_manifest/final_commission_file_manifest.xlsx
Output directory: /home/edik/projects/ec_cjeu/output/final_manifest
Snippet directory: /home/edik/projects/ec_cjeu/logs/final_manifest/classification_inputs
Output workbook: /home/edik/projects/ec_cjeu/output/final_manifest/final_commission_manifest_classified.xlsx
Checkpoint: /home/edik/projects/ec_cjeu/logs/final_manifest/final_commission_manifest_classified.jsonl
Model: google/gemma-4-31B-it
RUN_MODE: update
CONCURRENT_REQUESTS: 4
Snippet policy: first 15,000 + last 10,000 characters


## 2. Load the final manifest and identify eligible document rows

Only rows with a downloaded/already-present source file, a successful or reviewable processing status, and a non-empty `text_path` are sent to Gemma. Failed download attempts and source entries without a processed document remain in the final workbook but are not inference jobs.


In [2]:
def clean_scalar(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    text = str(value).strip()
    # Excel dates sometimes arrive as midnight timestamps.
    if re.fullmatch(r"\d{4}-\d{2}-\d{2} 00:00:00", text):
        return text[:10]
    return text


manifest = pd.read_excel(MANIFEST_PATH, dtype=object).fillna("")
manifest.columns = [str(c).strip() for c in manifest.columns]

required_columns = {
    "manifest_row_id",
    "source",
    "date",
    "case_numbers",
    "celex",
    "title",
    "document_class",
    "source_url",
    "download_status",
    "processing_status",
    "text_path",
}
missing = sorted(required_columns - set(manifest.columns))
if missing:
    raise ValueError(f"Manifest is missing required columns: {missing}")
if manifest["manifest_row_id"].astype(str).str.strip().eq("").any():
    raise ValueError("manifest_row_id contains blank values.")
if manifest["manifest_row_id"].duplicated().any():
    raise ValueError("manifest_row_id must be unique before classification.")

for col in manifest.columns:
    manifest[col] = manifest[col].map(clean_scalar)

eligible_mask = (
    manifest["download_status"].isin({"downloaded", "already_present"})
    & manifest["processing_status"].isin({"success", "review"})
    & manifest["text_path"].ne("")
)

eligible = manifest.loc[eligible_mask].copy()

print(f"Manifest rows: {len(manifest):,}")
print(f"Eligible processed documents: {len(eligible):,}")
print(f"Not sent to Gemma: {len(manifest) - len(eligible):,}")
print("\nEligible rows by source:")
print(eligible["source"].value_counts(dropna=False).to_string())
print("\nProcessing status:")
print(eligible["processing_status"].value_counts(dropna=False).to_string())


Manifest rows: 3,913
Eligible processed documents: 3,582
Not sent to Gemma: 331

Eligible rows by source:
source
com_legacy      1579
com_registry    1150
com_dbcomp       837
com_manual        16

Processing status:
processing_status
success    3571
review       11


/tmp/ipykernel_1063182/3825659545.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  manifest = pd.read_excel(MANIFEST_PATH, dtype=object).fillna("")


## 3. Resolve processed-text paths and create Gemma classification inputs

Every eligible processed document is converted **before inference** into one persistent text input under:

`logs/final_manifest/classification_inputs/`

That file is the exact text sent to Gemma: the first **15,000 characters** and the last **10,000 characters**. If the complete document is shorter than 25,000 characters, the full text is saved once rather than duplicating overlapping head/tail text.


In [3]:

def resolve_text_path(path_value: str) -> Path | None:
    value = clean_scalar(path_value)
    if not value:
        return None

    raw = Path(value).expanduser()
    candidates: list[Path] = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([
            PROJECT_ROOT / raw,
            PROJECT_ROOT.parent / raw,
            NOTEBOOK_DIR / raw,
            raw,
        ])

        if raw.parts and raw.parts[0] == PROJECT_ROOT.name:
            candidates.append(PROJECT_ROOT.parent / raw)
        if raw.parts and raw.parts[0] in {"eccjeu", "ec_cjeu"}:
            candidates.append(PROJECT_ROOT.parent / raw)

    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.is_file():
            return candidate.resolve()
    return None


def safe_filename(value: str, fallback: str = "document") -> str:
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", clean_scalar(value))
    text = re.sub(r"_+", "_", text).strip("_.-")
    return text or fallback


def make_classification_snippet(text: str) -> tuple[str, str]:
    """
    Return (snippet_text, strategy).

    Short documents are kept whole.
    Long documents are reduced to the fixed first 15k + last 10k characters.
    """
    limit = HEAD_CHARS + TAIL_CHARS
    if len(text) <= limit:
        return text, "full_under_25k"

    snippet = (
        "[BEGINNING OF PROCESSED DOCUMENT]\n"
        + text[:HEAD_CHARS]
        + "\n\n[... MIDDLE OMITTED ...]\n\n"
        + "[ENDING OF PROCESSED DOCUMENT]\n"
        + text[-TAIL_CHARS:]
    )
    return snippet, "head_15k_tail_10k"


def prepare_document_input(row: pd.Series) -> dict[str, Any]:
    resolved = resolve_text_path(row["text_path"])

    base = {
        "manifest_row_id": row["manifest_row_id"],
        "source": row["source"],
        "manifest_text_path": row["text_path"],
        "resolved_text_path": str(resolved) if resolved else "",
        "input_status": "",
        "input_error": "",
        "input_strategy": "",
        "original_document_chars": None,
        "prompt_document_chars": None,
        "llm_input_path": "",
    }

    if resolved is None:
        return {
            **base,
            "input_status": "error",
            "input_error": "Processed text path could not be resolved",
        }

    try:
        text = resolved.read_text(encoding="utf-8", errors="replace")
    except Exception as exc:
        return {
            **base,
            "input_status": "error",
            "input_error": f"{type(exc).__name__}: {exc}",
        }

    if not text.strip():
        return {
            **base,
            "input_status": "error",
            "input_error": "Processed text file is empty",
        }

    snippet, strategy = make_classification_snippet(text)

    snippet_name = safe_filename(row["manifest_row_id"]) + "__classification_input.txt"
    snippet_path = SNIPPET_DIR / snippet_name
    snippet_path.write_text(snippet, encoding="utf-8")

    return {
        **base,
        "input_status": "ready",
        "input_strategy": strategy,
        "original_document_chars": len(text),
        "prompt_document_chars": len(snippet),
        "llm_input_path": str(snippet_path.resolve()),
    }


# Create/refresh snippets for every eligible document BEFORE any Gemma request.
input_rows = [prepare_document_input(row) for _, row in eligible.iterrows()]
input_audit = pd.DataFrame(input_rows)

print(input_audit["input_status"].value_counts(dropna=False).to_string())
print("\nInput strategy:")
print(input_audit["input_strategy"].value_counts(dropna=False).to_string())
print(f"\nClassification inputs saved to: {SNIPPET_DIR}")

problems = input_audit[input_audit["input_status"] != "ready"]
print(f"Input-resolution problems: {len(problems):,}")
if len(problems):
    display(problems.head(20))


input_status
ready    3582

Input strategy:
input_strategy
head_15k_tail_10k    2236
full_under_25k       1346

Classification inputs saved to: /home/edik/projects/ec_cjeu/logs/final_manifest/classification_inputs
Input-resolution problems: 0



## 4. Fast Gemma extraction prompt

Gemma receives:

1. the metadata already present in the final manifest as a **hint**; and
2. only the persistent head/tail classification snippet.

The model does **not** produce evidence, confidence scores, explanations, or field-by-field assessments. It returns only candidate values that appear to identify **this document**.


In [4]:
RESPONSE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "manifest_row_id": {"type": "string"},
        "case_numbers": {
            "type": "array",
            "items": {"type": "string"},
        },
        "document_dates": {
            "type": "array",
            "items": {"type": "string"},
        },
        "celex_numbers": {
            "type": "array",
            "items": {"type": "string"},
        },
        "published_decision_references": {
            "type": "array",
            "items": {"type": "string"},
        },
    },
    "required": [
        "manifest_row_id",
        "case_numbers",
        "document_dates",
        "celex_numbers",
        "published_decision_references",
    ],
    "additionalProperties": False,
}


SYSTEM_PROMPT = """
You extract identity metadata from European Commission competition documents.
Return only the requested JSON. Be conservative: return a candidate only when
the supplied document input indicates that it identifies THIS document, rather
than another decision or legal act merely cited in the text.
""".strip()


PROMPT_RULES = r"""
Identify the metadata of THIS Commission document.

You are given CURRENT METADATA from the dataset. Treat it as a useful hint,
but it may be incomplete or wrong. Inspect the supplied document input and
return only candidates that appear to characterise THIS document.

Return ONLY these candidate families:

1. case_numbers
   Competition case/procedure numbers belonging to THIS document.
   Common forms include:
   IV/61
   IV/26.760
   IV/95
   COMP/37.792
   COMP/38.899
   COMP/F-2/32432
   COMP/F2/99912
   AT.39904
   AT.40099

2. document_dates
   Date(s) that characterise THIS document, especially the decision/adoption/
   publication/closure/announcement date. Normalise complete dates to YYYY-MM-DD.
   Do not return dates that clearly belong only to cited or historical events.

3. celex_numbers
   CELEX identifiers explicitly present in the supplied text, commonly like
   31964D0233. NEVER construct or infer a CELEX number.

4. published_decision_references
   Formal/published Commission decision references identifying THIS document,
   e.g. 233/64/EEC, 80/917/EEC, 98/531/EC, 2004/337/EC and analogous
   EEC/EC/EU/ECSC/Euratom forms.

IMPORTANT:
- Do not list every identifier or date mentioned in the text.
- Do not return references that merely identify another Commission decision,
  Court case, Regulation, Directive, or legal act.
- The beginning and end of the document are intentionally supplied because
  these identity fields normally appear there.
- If no qualifying candidate is found for a field, return an empty array.
- No explanations, evidence, confidence scores, or commentary.
""".strip()


def metadata_for_prompt(row: pd.Series) -> dict[str, Any]:
    fields = [
        "manifest_row_id",
        "source",
        "source_record_id",
        "decision_id",
        "date",
        "case_numbers",
        "title",
        "parties",
        "celex",
        "document_class",
        "source_url",
        "downloaded_url",
        "source_language",
    ]
    return {
        field: clean_scalar(row.get(field, ""))
        for field in fields
        if field in row.index
    }


def build_user_prompt(row: pd.Series, document_text: str, input_strategy: str) -> str:
    metadata_json = json.dumps(metadata_for_prompt(row), ensure_ascii=False, indent=2)

    return f"""
{PROMPT_RULES}

CURRENT METADATA / OUR CURRENT BELIEF
{metadata_json}

DOCUMENT INPUT
<<<BEGIN INPUT>>>
{document_text}
<<<END INPUT>>>

Return the JSON candidates for manifest_row_id={clean_scalar(row.get("manifest_row_id"))}.
""".strip()


print("Fast structured schema prepared.")


Fast structured schema prepared.



## 5. Build stable inference jobs

The checkpoint signature includes the prompt version, current metadata, processed-text identity, and snippet strategy. Changing the prompt, metadata, or processed text automatically makes the corresponding old result stale.


In [5]:
audit_by_id = input_audit.set_index("manifest_row_id").to_dict("index")


def stable_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def make_input_signature(row: pd.Series, audit: dict[str, Any]) -> str:
    snippet_path = Path(audit["llm_input_path"])
    snippet_sha256 = hashlib.sha256(snippet_path.read_bytes()).hexdigest()

    payload = {
        "prompt_version": PROMPT_VERSION,
        "manifest_row_id": row["manifest_row_id"],
        "metadata": metadata_for_prompt(row),
        "snippet_sha256": snippet_sha256,
        "input_strategy": audit.get("input_strategy", ""),
        "original_document_chars": audit.get("original_document_chars"),
        "prompt_document_chars": audit.get("prompt_document_chars"),
    }
    return hashlib.sha256(stable_json(payload).encode("utf-8")).hexdigest()


jobs: list[dict[str, Any]] = []
for _, row in eligible.iterrows():
    rid = row["manifest_row_id"]
    audit = audit_by_id[rid]
    if audit.get("input_status") != "ready":
        continue

    signature = make_input_signature(row, audit)
    jobs.append({
        "job_id": rid,
        "manifest_row_id": rid,
        "source": row["source"],
        "input_signature": signature,
        "llm_input_path": audit["llm_input_path"],
        "input_strategy": audit["input_strategy"],
        "original_document_chars": int(audit["original_document_chars"]),
        "prompt_document_chars": int(audit["prompt_document_chars"]),
        "row_metadata": metadata_for_prompt(row),
    })

if TEST_LIMIT is not None:
    jobs = jobs[:TEST_LIMIT]

print(f"Prepared inference jobs: {len(jobs):,}")
print(pd.Series([j["input_strategy"] for j in jobs]).value_counts().to_string())


Prepared inference jobs: 3,582
head_15k_tail_10k    2236
full_under_25k       1346


## 6. vLLM call, response validation, and JSONL checkpoints

Inference is parallelised with `asyncio` + `httpx.AsyncClient` + `asyncio.Semaphore`. Results are written sequentially as tasks complete, so model calls stay concurrent while checkpoint writes remain safe.


In [6]:

def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def extract_json_object(raw: str) -> dict[str, Any]:
    text = clean_scalar(raw)
    if not text:
        raise ValueError("Empty model response")
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start < 0 or end <= start:
            raise
        parsed = json.loads(text[start:end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("Model response is not a JSON object")
    return parsed


def validate_response(parsed: dict[str, Any], manifest_row_id: str) -> dict[str, Any]:
    required = {
        "manifest_row_id",
        "case_numbers",
        "document_dates",
        "celex_numbers",
        "published_decision_references",
    }
    missing = sorted(required - set(parsed))
    if missing:
        raise ValueError(f"Structured response missing fields: {missing}")

    if clean_scalar(parsed.get("manifest_row_id")) != manifest_row_id:
        raise ValueError(
            f"Response manifest_row_id={parsed.get('manifest_row_id')!r} "
            f"does not match {manifest_row_id!r}"
        )

    for key in [
        "case_numbers",
        "document_dates",
        "celex_numbers",
        "published_decision_references",
    ]:
        if not isinstance(parsed.get(key), list):
            raise ValueError(f"{key} must be a list")

    return parsed


async def call_vllm(client: httpx.AsyncClient, prompt: str) -> dict[str, Any]:
    url = VLLM_BASE_URL.rstrip("/") + "/chat/completions"
    headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_OUTPUT_TOKENS,
        "structured_outputs": {
            "json": RESPONSE_JSON_SCHEMA,
        },
    }

    response = await client.post(url, headers=headers, json=payload)
    response.raise_for_status()
    data = response.json()
    choice = data["choices"][0]
    return {
        "content": choice["message"]["content"],
        "finish_reason": choice.get("finish_reason", ""),
        "usage": data.get("usage", {}),
    }


def load_jsonl_records(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []

    records: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f"Skipping invalid JSONL line {line_number}: {exc}")
    return records


def append_jsonl_record(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()


def latest_checkpoint_records(records: list[dict[str, Any]]) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    for record in records:
        job_id = clean_scalar(record.get("job_id")) or clean_scalar(record.get("manifest_row_id"))
        if not job_id:
            continue
        previous = latest.get(job_id)
        if previous is None or clean_scalar(record.get("classified_at")) >= clean_scalar(previous.get("classified_at")):
            latest[job_id] = record
    return latest


async def classify_one_job(
    job: dict[str, Any],
    client: httpx.AsyncClient,
    semaphore: asyncio.Semaphore,
) -> dict[str, Any]:
    common = {
        "job_id": job["job_id"],
        "manifest_row_id": job["manifest_row_id"],
        "source": job["source"],
        "input_signature": job["input_signature"],
        "prompt_version": PROMPT_VERSION,
        "model": MODEL,
        "llm_input_path": job["llm_input_path"],
        "input_strategy": job["input_strategy"],
        "original_document_chars": job["original_document_chars"],
        "prompt_document_chars": job["prompt_document_chars"],
    }

    try:
        document_text = Path(job["llm_input_path"]).read_text(
            encoding="utf-8", errors="replace"
        )
    except Exception as exc:
        return {
            **common,
            "llm_status": "error",
            "llm_error": f"Could not read classification snippet: {type(exc).__name__}: {exc}",
            "classified_at": utc_now_iso(),
            "attempts": 0,
            "finish_reason": "",
            "usage": {},
            "raw_response": "",
            "parsed_response": None,
        }

    if len(document_text) != job["prompt_document_chars"]:
        return {
            **common,
            "llm_status": "error",
            "llm_error": (
                "Classification snippet changed after job construction: expected "
                f"{job['prompt_document_chars']:,} chars, read {len(document_text):,}"
            ),
            "classified_at": utc_now_iso(),
            "attempts": 0,
            "finish_reason": "",
            "usage": {},
            "raw_response": "",
            "parsed_response": None,
        }

    row = manifest.loc[
        manifest["manifest_row_id"].eq(job["manifest_row_id"])
    ].iloc[0]
    prompt = build_user_prompt(row, document_text, job["input_strategy"])

    last_error = ""
    last_raw = ""
    last_finish = ""
    last_usage: dict[str, Any] = {}

    async with semaphore:
        for attempt in range(1, MAX_RETRIES + 2):
            try:
                result = await call_vllm(client, prompt)
                last_raw = result["content"]
                last_finish = clean_scalar(result.get("finish_reason"))
                last_usage = result.get("usage", {}) or {}

                if last_finish == "length":
                    raise ValueError(
                        f"Model output reached MAX_OUTPUT_TOKENS={MAX_OUTPUT_TOKENS:,}"
                    )

                parsed = extract_json_object(last_raw)
                parsed = validate_response(parsed, job["manifest_row_id"])

                return {
                    **common,
                    "llm_status": "ok",
                    "llm_error": "",
                    "classified_at": utc_now_iso(),
                    "attempts": attempt,
                    "finish_reason": last_finish,
                    "usage": last_usage,
                    "raw_response": last_raw,
                    "parsed_response": parsed,
                }

            except Exception as exc:
                last_error = f"{type(exc).__name__}: {exc}"
                if attempt <= MAX_RETRIES:
                    await asyncio.sleep(min(2 ** attempt, 10))

    return {
        **common,
        "llm_status": "error",
        "llm_error": last_error,
        "classified_at": utc_now_iso(),
        "attempts": MAX_RETRIES + 1,
        "finish_reason": last_finish,
        "usage": last_usage,
        "raw_response": last_raw,
        "parsed_response": None,
    }


print("vLLM functions ready.")


vLLM functions ready.


## 7. Run the parallel classification

This is the only cell that sends inference requests. Because results are appended to the JSONL checkpoint as each task finishes, an interrupted run can resume safely with `RUN_MODE="update"`.


In [ ]:
async def run_document_jobs(jobs: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if RUN_MODE == "redo_all" and CHECKPOINT_JSONL.exists():
        CHECKPOINT_JSONL.unlink()

    existing = load_jsonl_records(CHECKPOINT_JSONL)
    latest = latest_checkpoint_records(existing)

    if RUN_MODE == "rebuild_excel":
        print("RUN_MODE='rebuild_excel': no LLM requests will be made.")
        return existing

    pending: list[dict[str, Any]] = []
    reused = 0
    stale = 0

    for job in jobs:
        previous = latest.get(job["job_id"])
        if (
            RUN_MODE == "update"
            and previous is not None
            and previous.get("llm_status") == "ok"
            and clean_scalar(previous.get("input_signature")) == job["input_signature"]
        ):
            reused += 1
            continue
        if previous is not None and clean_scalar(previous.get("input_signature")) != job["input_signature"]:
            stale += 1
        pending.append(job)

    print(f"Successful current checkpoints reused: {reused:,}")
    print(f"Stale checkpoints to refresh: {stale:,}")
    print(f"LLM jobs this run: {len(pending):,}")

    if not pending:
        return load_jsonl_records(CHECKPOINT_JSONL)

    semaphore = asyncio.Semaphore(CONCURRENT_REQUESTS)
    timeout = httpx.Timeout(REQUEST_TIMEOUT_SECONDS)

    async with httpx.AsyncClient(timeout=timeout) as client:
        tasks = [
            asyncio.create_task(classify_one_job(job, client, semaphore))
            for job in pending
        ]

        progress = tqdm(total=len(tasks), desc="Gemma manifest classification", unit="document", dynamic_ncols=True)
        try:
            for task in asyncio.as_completed(tasks):
                record = await task
                # Writes are intentionally sequential here even though inference is concurrent.
                append_jsonl_record(CHECKPOINT_JSONL, record)
                progress.set_postfix_str(
                    f"{record['manifest_row_id']} → {record['llm_status']}"
                )
                progress.update(1)
        finally:
            progress.close()

    return load_jsonl_records(CHECKPOINT_JSONL)


classification_records = await run_document_jobs(jobs)
latest_after_run = latest_checkpoint_records(classification_records)

current_job_ids = {job["job_id"] for job in jobs}
current_latest = {
    job_id: record
    for job_id, record in latest_after_run.items()
    if job_id in current_job_ids
}

print("\nCurrent checkpoint status:")
print(pd.Series(
    [record.get("llm_status", "missing") for record in current_latest.values()],
    dtype="object",
).value_counts(dropna=False).to_string())


Successful current checkpoints reused: 0
Stale checkpoints to refresh: 0
LLM jobs this run: 3,582


Gemma manifest classification:   0%|          | 0/3582 [00:00<?, ?document/s]


## 8. Flatten the candidate JSON

The checkpoint JSONL remains the raw machine-readable record. For Excel/CSV, candidate arrays are flattened into semicolon-separated columns.


In [ ]:
def ordered_unique(values: list[Any]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []

    for value in values:
        value = clean_scalar(value)
        if value and value not in seen:
            seen.add(value)
            out.append(value)
    return out


def join_candidates(values: Any) -> str:
    if not isinstance(values, list):
        return ""
    return "; ".join(ordered_unique(values))


def flatten_record(
    record: dict[str, Any] | None,
    job: dict[str, Any],
) -> dict[str, Any]:

    base = {
        "manifest_row_id": job["manifest_row_id"],
        "gemma_input_strategy": job["input_strategy"],
    }

    if record is None:
        return {
            **base,
            "gemma_case_numbers": "",
            "gemma_document_dates": "",
            "gemma_celex": "",
            "gemma_published_decision_references": "",
        }

    if (
        record.get("llm_status") != "ok"
        or not isinstance(record.get("parsed_response"), dict)
    ):
        return {
            **base,
            "gemma_case_numbers": "",
            "gemma_document_dates": "",
            "gemma_celex": "",
            "gemma_published_decision_references": "",
        }

    parsed = record["parsed_response"]

    return {
        **base,
        "gemma_case_numbers": join_candidates(parsed.get("case_numbers")),
        "gemma_document_dates": join_candidates(parsed.get("document_dates")),
        "gemma_celex": join_candidates(parsed.get("celex_numbers")),
        "gemma_published_decision_references": join_candidates(
            parsed.get("published_decision_references")
        ),
    }


latest = latest_checkpoint_records(load_jsonl_records(CHECKPOINT_JSONL))

flattened_rows = []
for job in jobs:
    record = latest.get(job["job_id"])

    if (
        record is not None
        and clean_scalar(record.get("input_signature")) != job["input_signature"]
    ):
        record = None

    flattened_rows.append(flatten_record(record, job))

gemma_df = pd.DataFrame(flattened_rows)

print(f"Flattened current jobs: {len(gemma_df):,}")



## 9. Merge candidate fields back onto the complete manifest

All original manifest rows remain. The Gemma columns are additive; no source metadata is overwritten.


In [ ]:
classified = manifest.copy()

classified = classified.merge(
    gemma_df,
    on="manifest_row_id",
    how="left",
    validate="one_to_one",
)

GEMMA_OUTPUT_COLUMNS = [
    "gemma_input_strategy",
    "gemma_case_numbers",
    "gemma_document_dates",
    "gemma_celex",
    "gemma_published_decision_references",
]

for column in GEMMA_OUTPUT_COLUMNS:
    if column not in classified.columns:
        classified[column] = ""

print(f"Final manifest rows: {len(classified):,}")
print("Gemma columns added:")
for column in GEMMA_OUTPUT_COLUMNS:
    print(" -", column)


## 10. Export the classified final manifest

Human-facing output stays in `output/final_manifest/`:

- `final_commission_manifest_classified.xlsx`

Gemma checkpoints and intermediate classification inputs stay under `logs/final_manifest/`.

The Excel workbook contains only:

- `files` — the full manifest plus the five Gemma columns;
- `summary` — compact counts by source.


In [ ]:
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

summary_rows = []

for source, group in classified.groupby("source", dropna=False):
    summary_rows.append(
        {
            "source": source,
            "rows": len(group),
            "gemma_case_numbers_found": int(group["gemma_case_numbers"].astype(str).str.strip().ne("").sum()),
            "gemma_document_dates_found": int(group["gemma_document_dates"].astype(str).str.strip().ne("").sum()),
            "gemma_celex_found": int(group["gemma_celex"].astype(str).str.strip().ne("").sum()),
            "gemma_published_decision_references_found": int(
                group["gemma_published_decision_references"].astype(str).str.strip().ne("").sum()
            ),
        }
    )

summary_df = pd.DataFrame(summary_rows)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tmp_xlsx = OUTPUT_XLSX.with_name(OUTPUT_XLSX.stem + ".pending.xlsx")

with pd.ExcelWriter(
    tmp_xlsx,
    engine="openpyxl",
    datetime_format="yyyy-mm-dd",
) as writer:

    classified.to_excel(writer, sheet_name="files", index=False)
    summary_df.to_excel(writer, sheet_name="summary", index=False)

    header_fill = PatternFill("solid", fgColor="19334D")
    header_font = Font(color="FFFFFF", bold=True)
    gemma_fill = PatternFill("solid", fgColor="D9EAF7")

    for sheet_name in ["files", "summary"]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        ws.row_dimensions[1].height = 30

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font

    ws = writer.sheets["files"]

    for idx, col_name in enumerate(classified.columns, start=1):
        letter = get_column_letter(idx)

        if col_name in {"title", "parties"}:
            width = 42
        elif col_name.endswith("_path") or col_name.endswith("_url"):
            width = 50
        else:
            width = min(max(len(col_name) + 2, 14), 34)

        ws.column_dimensions[letter].width = width

        if col_name.startswith("gemma_"):
            for row_no in range(2, len(classified) + 2):
                ws.cell(row_no, idx).fill = gemma_fill

tmp_xlsx.replace(OUTPUT_XLSX)

print(f"Wrote {len(classified):,} manifest rows to {OUTPUT_XLSX}")
print(f"Checkpoint JSONL: {CHECKPOINT_JSONL}")
print(f"Classification inputs: {SNIPPET_DIR}")


## 11. Quick QA view

A compact view of source metadata beside the five Gemma output columns.


In [ ]:
qa_columns = [
    c for c in [
        "manifest_row_id",
        "source",
        "title",
        "date",
        "gemma_document_dates",
        "case_numbers",
        "gemma_case_numbers",
        "celex",
        "gemma_celex",
        "gemma_published_decision_references",
        "gemma_input_strategy",
    ]
    if c in classified.columns
]

display(classified[qa_columns].head(50))
